# 2025-010D: a Falcon 9 second stage, community-driven astrometry and a lunar impact

On 2026-08-05 the Falcon 9 second stage from the January 2025 Blue Ghost / HAKUTO-R launch struck the Moon near crater Einstein. It was the first lunar impactor whose provenance, mass and dimensions were known in advance. This notebook fits the final nine-night arc of the *public* astrometry (compiled by Project Pluto, public domain), measures the area-to-mass ratio, propagates into the Moon, and scores the impact site against the crater NASA's Lunar Reconnaissance Orbiter imaged.

## Setup

On Google Colab (or any fresh environment), the first
`empyrean.initialize()` call pays a one-time ~740 MB kernel
download.

In [1]:
%pip install -q empyrean==0.11.0rc1

In [2]:
import re
from pathlib import Path

import empyrean
import numpy as np
from empyrean import (
    CartesianOrbits,
    CovarianceKind,
    Epochs,
    EventConfig,
    JointFunctional,
    JointProvenance,
    JointQuality,
    ODConfig,
    Origin,
    PackedJoint,
    PackedJoints,
    SolveFor,
    SRPParams,
    TimeScale,
)
from empyrean.od.result import (
    OriginPolicy,
    OriginPolicyMode,
    WeightingConfig,
    WeightingPreset,
)

# The 402 public observations, in ADES PSV, from Bill Gray's Project Pluto pages
# (public domain; the MPC does not distribute this object's astrometry). Download
# https://raw.githubusercontent.com/Empyrean-Dynamics/empyrean-scenarios/main/2025-010D/astrometry.psv
# and save it as astrometry.psv in the directory you run from.
ASTROMETRY = next(
    p for p in (Path("astrometry.psv"), Path("2025-010D/astrometry.psv")) if p.exists()
)
# Gray's fit to the same 74 public observations (Project Pluto pseudo-MPEC 25010d_new).
GRAY_AMR, GRAY_AMR_SIGMA = 0.0079, 0.0017  # m^2/kg
# The crater NASA's Lunar Reconnaissance Orbiter imaged on 2026-08-11/12 (NASA,
# "NASA's LRO Images Falcon 9 Crater on Moon", science.nasa.gov), and Gray's final
# prediction of 2026-08-01 from his full record, which is not public (Project Pluto,
# https://www.projectpluto.com/25010d.htm). Selenographic degrees north and east.
CRATER = (19.4759, 266.7138)
GRAY_FINAL = (19.461, 266.707)
MOON_RADIUS_KM = 1737.4  # the Moon's mean radius


def separation_km(a, b):
    """Great-circle distance between two (lat, lon) points in degrees, on the Moon."""
    p1, p2 = np.radians(a[0]), np.radians(b[0])
    dl = np.radians(b[1] - a[1])
    h = np.sin((p2 - p1) / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return 2.0 * MOON_RADIUS_KM * np.arcsin(np.sqrt(h))


def _prior_joint(state6, param_vars, present_bits):
    """Block-diagonal refine prior: the fit's state 6x6 plus one diagonal
    prior variance per parameter axis. The AMRAT prior rides the orbit's one
    packed joint now (bit 4 of the axis table). The removed
    ``SRPParams.amrat_variance`` column is gone."""
    width = 6 + len(param_vars)
    full = np.zeros((width, width), dtype=np.float64)
    full[:6, :6] = np.asarray(state6, dtype=np.float64)
    for k, v in enumerate(param_vars):
        full[6 + k, 6 + k] = v
    tri = [full[i, j] for i in range(width) for j in range(i + 1)]
    return PackedJoint(
        layout_present=present_bits,
        layout_considered=0,
        layout_marginalized=0,
        width=width,
        tri=np.asarray(tri, dtype=np.float64),
        kind=CovarianceKind.LINEAR,
        quality=JointQuality.POSITIVE_DEFINITE,
        cov_rank=0,
        cov_null_axes=0,
        cov_min_eig_corr=float("nan"),
        functional=JointFunctional.STATE,
        provenance=JointProvenance.DERIVED,
        asserted_source=None,
    )


empyrean.initialize()

## 1. The committed public astrometry

402 observations compiled by Project Pluto (Bill Gray), public domain, transcribed into the ADES PSV the reader consumes. The final arc: nine nights, five stations, four continents.

In [3]:
# No MPC query: these 402 observations are the
# public-domain record compiled by Project Pluto (see README).
obs, _ = empyrean.read_ades(str(ASTROMETRY))
print(f"{len(obs)} public observations (Project Pluto, public domain)")

# Final pre-impact arc: nine nights, five stations, four continents.
iso = obs.obs_time.to_pylist()
final_arc = obs.apply_mask([t >= "2026-07-01" for t in iso])
print(f"{len(final_arc)} in the final arc (2026-07-23 → 08-01)")

402 public observations (Project Pluto, public domain)
74 in the final arc (2026-07-23 → 08-01)


## 2. Geocentric fit + AMR refine

`EXPLICIT`/`EARTH` origin policy (heliocentric Gauss is unphysical for an Earth-orbiting stage), residual-matched 0.3 arcsecond weighting (the file carries no per-observation sigmas), then the refine that measures the area-to-mass ratio, against Gray's value on the same arc.

In [4]:
# EXPLICIT/EARTH: heliocentric Gauss IOD is unphysical for a
# catalogued Earth-orbiting object.
geocentric = ODConfig(
    origin=OriginPolicy(mode=OriginPolicyMode.EXPLICIT, origin=Origin.EARTH),
    # The PSV carries no per-observation sigmas and these community
    # stations have no survey-weighting rules, so the catalog
    # fallback would assume ~1.5" for astrometry that fits at 0.3".
    # State the residual-matched sigma explicitly.
    weighting=WeightingConfig(preset=WeightingPreset.NONE, default_sigma_arcsec=0.3),
)
fit = empyrean.determine(final_arc, config=geocentric).single()
s = fit.summary
print(
    f"fit: chi2/dof {s.reduced_chi2:.3f}  "
    f'RMS RA·cos(d) {s.rms_ra_arcsec:.3f}" Dec {s.rms_dec_arcsec:.3f}"  '
    f"({s.num_selected}/{s.num_obs} obs)"
)

# AMRAT is a refine-path solve: prime the fitted orbit with an SRP
# prior wide enough for the artsat regime (~1e-2 m²/kg, 100× any
# asteroid) so the astrometry, not the prior, drives the fit.
# The AMRAT prior rides the fitted orbit's packed joint (its AMRAT axis,
# bit 4), widening the state 6x6 to 7. The removed `amrat_variance`
# column is gone.
srp = SRPParams.from_kwargs(amrat=[0.008], cr=[1.0])
state6 = fit.orbit.covariance.joint(0).state_block()
amrat_prior = _prior_joint(state6, [1.0e-4], present_bits=1 << 4)
primed = CartesianOrbits.from_kwargs(
    orbit_id=fit.orbit.orbit_id.to_pylist(),
    object_id=fit.orbit.object_id.to_pylist(),
    coordinates=fit.orbit.coordinates,
    srp=srp,
    covariance=PackedJoints.from_joints([amrat_prior]),
)
refined = empyrean.refine(
    primed,
    final_arc,
    config=ODConfig(
        solve_for_flags=SolveFor(amrat="solved"),
        origin=OriginPolicy(mode=OriginPolicyMode.EXPLICIT, origin=Origin.EARTH),
        weighting=WeightingConfig(
            preset=WeightingPreset.NONE, default_sigma_arcsec=0.3
        ),
    ),
)
sc = refined.solved_covariance
amrat = refined.orbit.srp.amrat.to_numpy(zero_copy_only=False)[0]
amrat_sigma = float(np.sqrt(sc.matrix[sc.amrat_slot, sc.amrat_slot]))
print(f"Fitted AMR = {amrat:.4f} ± {amrat_sigma:.4f} m²/kg")
print(f"Gray (same 74-obs arc): {GRAY_AMR} ± {GRAY_AMR_SIGMA} m²/kg")
gap = amrat - GRAY_AMR
print(
    f"Ours - Gray = {gap:+.4f} m²/kg, "
    f"{abs(gap) / np.hypot(amrat_sigma, GRAY_AMR_SIGMA):.2f} combined sigma"
)

fit: chi2/dof 0.540  RMS RA·cos(d) 0.323" Dec 0.279"  (74/74 obs)
Fitted AMR = 0.0029 ± 0.0011 m²/kg
Gray (same 74-obs arc): 0.0079 ± 0.0017 m²/kg
Ours - Gray = -0.0050 m²/kg, 2.45 combined sigma


## 3. Into the Moon

Propagate to 2026-08-05 and read the impact off the event stream: epoch and selenographic coordinates against Gray's fit, JPL's solution (which includes three radar Doppler measurements), and the crater LRO imaged.

In [5]:
epochs = Epochs.from_kwargs(
    mjd=[61250.0 + 0.05 * i for i in range(200)],
    scale=TimeScale.TDB.value,
)
prop = empyrean.propagate(
    refined.orbit, epochs, events=EventConfig(body_filter=[Origin.MOON])
)
imp = prop.events.impacts
assert len(imp) == 1, "expected one lunar impact"
for i in range(len(imp)):
    epoch_utc = (
        Epochs.from_mjd(
            [imp.epochs.mjd.to_numpy(zero_copy_only=False)[i]],
            scale=TimeScale.TDB.value,
        )
        .to_utc()
        .to_iso()[0]
    )
    # To 0.01 s: the fit's timing is no better than that.
    head, frac = epoch_utc.rstrip("Z").split(".")
    centis = round(float("0." + frac) * 100)
    assert centis < 100
    epoch_utc = f"{head}.{centis:02d}Z"
    lat = imp.latitude_deg.to_numpy(zero_copy_only=False)[i]
    lon = imp.longitude_deg.to_numpy(zero_copy_only=False)[i] % 360.0
    print(f"\nPredicted lunar impact (empyrean): {epoch_utc}")
    print(f"  at {lat:.3f}°N {lon:.3f}°E (selenographic)")
ours = (lat, lon)
print("Gray (74-obs arc):  2026-08-05T06:35:42.45Z at 19.577°N 266.630°E")
print("JPL #GA1A2/21:      2026-08-05T06:35:40Z ± 9 s at 19.507°N 266.7°E")
print("Confirmed:          2026-08-05 ~06:35 UTC near crater Einstein")
print(
    f"\nScored against the crater LRO imaged ({CRATER[0]}°N {CRATER[1]}°E), "
    f"great circle on a {MOON_RADIUS_KM} km Moon:"
)
print(f"  empyrean (this fit, public 74-obs arc)  {separation_km(ours, CRATER):.2f} km")
print(
    f"  Gray's final prediction (all data)      {separation_km(GRAY_FINAL, CRATER):.2f} km"
)


Predicted lunar impact (empyrean): 2026-08-05T06:35:40.17Z
  at 19.514°N 266.646°E (selenographic)
Gray (74-obs arc):  2026-08-05T06:35:42.45Z at 19.577°N 266.630°E
JPL #GA1A2/21:      2026-08-05T06:35:40Z ± 9 s at 19.507°N 266.7°E
Confirmed:          2026-08-05 ~06:35 UTC near crater Einstein

Scored against the crater LRO imaged (19.4759°N 266.7138°E), great circle on a 1737.4 km Moon:
  empyrean (this fit, public 74-obs arc)  2.27 km
  Gray's final prediction (all data)      0.49 km


## 4. Why the record is fit per arc

Score the final-arc orbit against the whole public record: the residuals climb five orders of magnitude through two lunar encounters and a tumbling body whose effective area changed between arcs (Gray 2026; Campbell et al. 2026).

In [6]:
# Score the final-arc orbit against the WHOLE public record. The
# residuals climb five orders of magnitude through two lunar
# encounters and a tumbling body whose effective area changed
# between arcs (Gray 2026; Campbell et al. 2026), consistent with
# the per-arc solutions Project Pluto published.
# evaluate (unlike determine) runs no rejection pass, so it refuses an
# observation whose station the engine cannot resolve rather than
# deselecting it (0.11.0 removed the old silent fallback). Drop any such
# unregistered community station explicitly (named by the engine's
# own refusal), so the scoring runs over the resolvable public record.
scorable = obs
while True:
    try:
        ev = empyrean.evaluate(refined.orbit, scorable)
        break
    except RuntimeError as exc:
        m = re.search(r'unknown observatory code "([^"]+)"', str(exc))
        if m is None:
            raise
        bad = m.group(1)
        scorable = scorable.apply_mask([s != bad for s in scorable.stn.to_pylist()])
        print(f"(dropped unregistered community station {bad} from scoring)")
r = ev.observations
t_mjd = r.epochs.mjd_tdb()
res_ra = r.ra_residual.to_numpy(zero_copy_only=False)
res_dec = r.dec_residual.to_numpy(zero_copy_only=False)
windows = [
    ("own arc   (2026 Jul-Aug)", 61244.0, 61300.0),
    ("Apr-May 2026", 61135.0, 61190.0),
    ("Dec 2025", 61020.0, 61040.0),
    ("Jan 2025 discovery", 60680.0, 60700.0),
]
print("\nFinal-arc orbit scored against the full 402-obs record:")
for label, lo, hi in windows:
    m = (t_mjd >= lo) & (t_mjd < hi)
    if m.any():
        rms = float(np.sqrt(np.nanmean(res_ra[m] ** 2 + res_dec[m] ** 2)))
        print(f'  {label:26s} RMS {rms:>12.2f}"')
print("(Five orders of magnitude across two lunar encounters and a")
print(" changing tumble, consistent with the per-arc solutions")
print(" Project Pluto published; Gray 2026, Campbell et al. 2026.)")

(dropped unregistered community station ScT from scoring)



Final-arc orbit scored against the full 402-obs record:
  own arc   (2026 Jul-Aug)   RMS         0.41"
  Apr-May 2026               RMS       165.29"
  Dec 2025                   RMS      1679.24"
  Jan 2025 discovery         RMS    470390.33"
(Five orders of magnitude across two lunar encounters and a
 changing tumble, consistent with the per-arc solutions
 Project Pluto published; Gray 2026, Campbell et al. 2026.)
